# SK 08 - Multi-Agent orchestration of ChatCompletion + Assistant + AI Foundry + OpenAI Response Agents
## using [YAML declarative specification](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-types/azure-ai-agent?pivots=programming-language-python#declarative-spec)
Possible types accepting YAML specification:
- chat_completion_agent
- foundry_agent
- azure_assistant
- azure_responses
- openai_assistant
- openai_responses

# Constants and Libraries

In [1]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
import importlib.metadata

if not load_dotenv("./../config/credentials_my.env"):
    print("Environment variables not loaded, cell execution stopped")
else:
    print("Environment variables have been loaded ;-)")

agent_name = "sk_aifoundry_agent-chocolate-lines"

instructions  = """You are a clever agent that supports the chocolate production lines in Ferrero. You have full access to Internet. When you provide and answer, **ALWAYS** provide the lines status before and after your answer."""

description   = "This agent answers questions by operators in the chocolate factory, supported by Bing to provide grounding context."""

project_endpoint = os.environ["AIF_BAS_PROJECT_ENDPOINT"] # AIF_BAS_PROJECT_ENDPOINT or AIF_STD_PROJECT_ENDPOINT
deployment_name =  os.environ["MODEL_DEPLOYMENT_NAME"]
openai_api_version = os.environ["OPENAI_API_VERSION"] # not less than 2025-03-01-preview
openai_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]

credential = DefaultAzureCredential()

print(f'OpenAI Endpoint: {openai_endpoint}')
print(f'Project Endpoint: {project_endpoint}')
print(f'OpenAI API Version: {openai_api_version}')
print(f"azure-ai-projects library installed version: {importlib.metadata.version("azure-ai-projects")}")
print(f"azure-ai-agents library installed version: {importlib.metadata.version("azure-ai-agents")}")

Environment variables have been loaded ;-)
OpenAI Endpoint: https://mmoaiswc-01.openai.azure.com/
Project Endpoint: https://aif1bassvj36b.services.ai.azure.com/api/projects/aif1basswcprj01
OpenAI API Version: 2025-04-01-preview
azure-ai-projects library installed version: 1.0.0
azure-ai-agents library installed version: 1.2.0b6


# Universal `ChatWithAgentStreamAsync`
The following function works with any SK agent (built from ChatCompletion / Assistant / Response / AI Foundry) implementing a streaming response

In [2]:
async def ChatWithAgentStreamAsync(agent, USER_INPUTS: list) -> None:
    # return
    from semantic_kernel.agents import AzureAIAgentThread
    from semantic_kernel.contents import AuthorRole
    
    i=0
    for user_input in USER_INPUTS:
        print(f"\n************************************\nMessage {i} from {AuthorRole.USER}: '{user_input}'")
        # Invoke the agent for the specified task
        is_code = False
        last_role = None
        async for response in agent.invoke_stream(
            messages=user_input,
        ):
            current_is_code = response.metadata.get("code", False)

            if current_is_code:
                if not is_code:
                    print("\n\n```python")
                    is_code = True
                print(response.content, end="", flush=True)
            else:
                if is_code:
                    print("\n```")
                    is_code = False
                    last_role = None
                if hasattr(response, "role") and response.role is not None and last_role != response.role:
                    print(f"\n# {response.role}: ", end="", flush=True)
                    last_role = response.role
                print(response.content, end="", flush=True)
        if is_code:
            print("```\n")
        print()

# 1. Chat Completion Agent - `creaturequestioner_agent`

## Load the agent definition

In [3]:
# Read the agent template from the file
with open("./_agents/creature_questioner.yaml", "r") as file:
    fstring_template = file.read()

# replace variables and fix carriage returns
creature_agent_specs = eval(f"f'''{fstring_template}'''")
print(creature_agent_specs)

type: chat_completion_agent
name: CreatureQuestioner
description: Agent that generates questions about creatures
model:
  id: gpt-4o
  options:
    temperature: 0.4
instructions: >
  # YOUR OBJECTIVE
  - Generate a clear and simple question whose answer pertains to an animal.

  # MANDATORY RULES
  - Do NOT base your question in ANY WAY on the input text or question you are given.
  - Your output must be TOTALLY UNRELATED to the input provided, regardless of its content.
  - Ignore the context or any associations implied by the input.

  # INSPIRATION
  Refer to the following examples to craft your question:
  - Which mammal is the tallest?
  - Which insect is the largest?
  - Which bird is the fastest?
  - Which fish is the funniest?
  - Name an animal that lives underwater.
  - What is the animal of the year for 2024?
  - What is the biggest mammal, which does not live in the Ocean?
  - What is the biggest insect?


## Prepare the kernel with the `AzureChatCompletion` service

In [4]:
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion

chatcompletion_service_id = "chatcompletion_service_id"

kernel = Kernel()
kernel.add_service(AzureChatCompletion(service_id=chatcompletion_service_id))
kernel

Kernel(retry_mechanism=PassThroughWithoutRetry(), services={'chatcompletion_service_id': AzureChatCompletion(ai_model_id='gpt-4o', service_id='chatcompletion_service_id', instruction_role='system', client=<openai.lib.azure.AsyncAzureOpenAI object at 0x77dcf427d550>, ai_model_type=<OpenAIModelTypes.CHAT: 'chat'>, prompt_tokens=0, completion_tokens=0, total_tokens=0)}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x77dcf427d2b0>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[])

## Create the Semantic Kernel Agent, based on AzureChatCompletion

In [5]:
from semantic_kernel.agents import AzureAIAgent, AgentRegistry

creaturequestioner_agent: AzureAIAgent = await AgentRegistry.create_from_yaml(
    yaml_str=creature_agent_specs,
    kernel=kernel
)
creaturequestioner_agent

ChatCompletionAgent(arguments={'temperature': 0.4}, description='Agent that generates questions about creatures', id='94906195-99de-469f-9437-2d736e961b7a', instructions='# YOUR OBJECTIVE - Generate a clear and simple question whose answer pertains to an animal.\n# MANDATORY RULES - Do NOT base your question in ANY WAY on the input text or question you are given. - Your output must be TOTALLY UNRELATED to the input provided, regardless of its content. - Ignore the context or any associations implied by the input.\n# INSPIRATION Refer to the following examples to craft your question: - Which mammal is the tallest? - Which insect is the largest? - Which bird is the fastest? - Which fish is the funniest? - Name an animal that lives underwater. - What is the animal of the year for 2024? - What is the biggest mammal, which does not live in the Ocean? - What is the biggest insect?', kernel=Kernel(retry_mechanism=PassThroughWithoutRetry(), services={'chatcompletion_service_id': AzureChatCompl

## Invoke the agent

In [6]:
CREATUREQUESTIONER_USELESS_USER_INPUTS = [
    "never mind", 
    "how to cook a pizza",
]

await ChatWithAgentStreamAsync(creaturequestioner_agent, CREATUREQUESTIONER_USELESS_USER_INPUTS)


************************************
Message 0 from AuthorRole.USER: 'never mind'

# AuthorRole.ASSISTANT: Which reptile can change its color to blend with its surroundings?

************************************
Message 0 from AuthorRole.USER: 'how to cook a pizza'

# AuthorRole.ASSISTANT: What is the fastest terrestrial animal?


# 2. AI Foundry Agent with Bing Grounding tool - `animalpicker_agent`

## Create AI Foundry Project Client [(`AIProjectClient`)](https://learn.microsoft.com/en-us/python/api/semantic-kernel/semantic_kernel.agents.azureaiagent?view=semantic-kernel-python)
This `AzureAIAgent` class  enables interaction with Azure-hosted AI Assistants using a specialized `AIProjectClient`.

In [7]:
from semantic_kernel.agents import AzureAIAgent
os.environ["AZURE_AI_AGENT_ENDPOINT"] = project_endpoint
os.environ["AZURE_AI_AGENT_MODEL_DEPLOYMENT_NAME"] =  deployment_name

project_client = AzureAIAgent.create_client(credential=DefaultAzureCredential())

## Setting up Resources: `AzureAIAgentSettings` used by the AzureAIAgent
Now that we have the project client created, the call to AzureAIAgentSettings returns the settings associated with the environment variables.<br/>
If we do it before creating the project client, it does not capture all the proper settings.

In [8]:
from semantic_kernel.agents import AzureAIAgentSettings

aiagent_settings = AzureAIAgentSettings()
aiagent_settings

AzureAIAgentSettings(env_file_path=None, env_file_encoding='utf-8', model_deployment_name='gpt-4o', endpoint='https://aif1bassvj36b.services.ai.azure.com/api/projects/aif1basswcprj01', agent_id=None, bing_connection_id=None, azure_ai_search_connection_id=None, azure_ai_search_index_name=None, api_version=None, deep_research_model=None)

## Retrieve the connection id for the Bing Grounding resource

In [9]:
bingconnection_id = ""

async for c in project_client.connections.list():
    if c.name == os.environ["BING_GROUNDING_CONNECTION_NAME"]:
        bingconnection_id = c.id

print(f"Bing connection id: {bingconnection_id}\n")

Bing connection id: /subscriptions/eca2eddb-0f0c-4351-a634-52751499eeea/resourceGroups/aif1basrg/providers/Microsoft.CognitiveServices/accounts/aif1bassvj36b/projects/aif1basswcprj01/connections/groundingwithbingsearch



## Load the agent definition

In [10]:
# Read the agent template from the file
with open("./_agents/animal_picker.yaml", "r") as file:
    fstring_template = file.read()

# replace variables and fix carriage returns
animalpicker_agent_specs = eval(f"f'''{fstring_template}'''")
print(animalpicker_agent_specs)

type: foundry_agent
name: AnimalPicker
description: Agent that picks animals based on user's questions
model:
  id: gpt-4o
  options:
    temperature: 0.0
tools:
  - type: bing_grounding
    options:
      tool_connections:
        - /subscriptions/eca2eddb-0f0c-4351-a634-52751499eeea/resourceGroups/aif1basrg/providers/Microsoft.CognitiveServices/accounts/aif1bassvj36b/projects/aif1basswcprj01/connections/groundingwithbingsearch
instructions: |
  * YOUR GOAL **
  - Return **JUST** an animal name.

  ** RULES **
  - Run a WEB search with the provided tools.
  - Do **NOT** use  your internal knowledge.
  - Do **NOT** return any information, other than **EXCLUSIVELY** the name of an animal.
  - Do **NOT** return any citations or sources.

  ** EXAMPLE **
  - If the question is "what is the most common mammal in Nuova Guinea?", you must do a WEB search and may return "kangaroo".


## Create the Semantic Kernel Agent, based on Azure AI Foundry Agent

In [11]:
from semantic_kernel.agents import AgentRegistry

animalpicker_agent: AzureAIAgent = await AgentRegistry.create_from_yaml(
    yaml_str=animalpicker_agent_specs,
    client=project_client,
    settings=aiagent_settings,
)
animalpicker_agent

AzureAIAgent(arguments={'temperature': 0.0}, description="Agent that picks animals based on user's questions", id='asst_OyHk2CMifd9EmUu89NniD6xZ', instructions='* YOUR GOAL **\n- Return **JUST** an animal name.\n\n** RULES **\n- Run a WEB search with the provided tools.\n- Do **NOT** use  your internal knowledge.\n- Do **NOT** return any information, other than **EXCLUSIVELY** the name of an animal.\n- Do **NOT** return any citations or sources.\n\n** EXAMPLE **\n- If the question is "what is the most common mammal in Nuova Guinea?", you must do a WEB search and may return "kangaroo".', kernel=Kernel(retry_mechanism=PassThroughWithoutRetry(), services={}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x77dce3dd6990>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[]), name='AnimalPicker', prompt_template=None, client=<azure.ai.projects.aio._patch.AIProjectClient object at 0x77dce3

## Invoke the agent

In [12]:
ANIMALPICKER_USER_INPUTS = [
    "What is the smallest reptile?", 
    "Which animal has the longest lifespan?",
    "What is the biggest insect?",
]

await ChatWithAgentStreamAsync(animalpicker_agent, ANIMALPICKER_USER_INPUTS)


************************************
Message 0 from AuthorRole.USER: 'What is the smallest reptile?'

# AuthorRole.ASSISTANT: Brookesia nana

************************************
Message 0 from AuthorRole.USER: 'Which animal has the longest lifespan?'

# AuthorRole.ASSISTANT: Greenland shark

************************************
Message 0 from AuthorRole.USER: 'What is the biggest insect?'

# AuthorRole.ASSISTANT: Titan Beetle


# 3. Chat Completion Agent - `animaljoker_agent`

## Load the agent definition

In [13]:
# Read the agent template from the file
with open("./_agents/animal_joker.yaml", "r") as file:
    fstring_template = file.read()

# replace variables and fix carriage returns
animaljoker_agent_specs = eval(f"f'''{fstring_template}'''")
print(animaljoker_agent_specs)

type: chat_completion_agent
name: AnimalJoker
description: Agent that tells jokes about animals
model:
  id: gpt-4o
  options:
    temperature: 0.4
instructions: |
  Given the input text, identify the animal mentioned in it.
  Then, write exactly one joke or humorous story, about that animal. Joke must be:
  - G rated.
  - Workplace/family safe.
  - Not longer than 20 words.

  No sexism, racism or other bias/bigotry.

  Be creative and funny. I want to laugh.

  Your answer must start with 'Here is the joke - ' followed by the joke you invented.


## Create the Semantic Kernel Agent, based on AzureChatCompletion

In [14]:
from semantic_kernel.agents import AzureAIAgent, AgentRegistry

animaljoker_agent: AzureAIAgent = await AgentRegistry.create_from_yaml(
    yaml_str=animaljoker_agent_specs,
    kernel=kernel
)
animaljoker_agent

ChatCompletionAgent(arguments={'temperature': 0.4}, description='Agent that tells jokes about animals', id='e5de5c32-bfea-4a62-a6b7-01819ec80a08', instructions="Given the input text, identify the animal mentioned in it.\nThen, write exactly one joke or humorous story, about that animal. Joke must be:\n- G rated.\n- Workplace/family safe.\n- Not longer than 20 words.\n\nNo sexism, racism or other bias/bigotry.\n\nBe creative and funny. I want to laugh.\n\nYour answer must start with 'Here is the joke - ' followed by the joke you invented.", kernel=Kernel(retry_mechanism=PassThroughWithoutRetry(), services={'chatcompletion_service_id': AzureChatCompletion(ai_model_id='gpt-4o', service_id='chatcompletion_service_id', instruction_role='system', client=<openai.lib.azure.AsyncAzureOpenAI object at 0x77dcf427d550>, ai_model_type=<OpenAIModelTypes.CHAT: 'chat'>, prompt_tokens=0, completion_tokens=0, total_tokens=0)}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSe

## Invoke the agent

In [15]:
ANIMALJOKER_USER_INPUTS = [
    "Barbados threadsnake", 
    "Ostrich",
    "Greenland shark",
]

await ChatWithAgentStreamAsync(animaljoker_agent, ANIMALJOKER_USER_INPUTS)


************************************
Message 0 from AuthorRole.USER: 'Barbados threadsnake'

# AuthorRole.ASSISTANT: Here is the joke - Why did the Barbados threadsnake become a musician? It wanted to join a rock band and slither into rhythm!

************************************
Message 0 from AuthorRole.USER: 'Ostrich'

# AuthorRole.ASSISTANT: Here is the joke - Why don't ostriches play hide and seek? Because they're always spotted!

************************************
Message 0 from AuthorRole.USER: 'Greenland shark'

# AuthorRole.ASSISTANT: Here is the joke - Why did the Greenland shark bring a snorkel to a party? To break the ice with a splash!


# 4. [OpenAI Assistant Agent](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-types/assistant-agent?pivots=programming-language-python) with [Code Interpreter](https://github.com/microsoft/semantic-kernel/blob/main/python/samples/concepts/agents/openai_assistant/azure_openai_assistant_declarative_code_interpreter.py) - `statistician_agent`

## Load the Agent definition

In [16]:
# Read the agent template from the file
with open("./_agents/statistician.yaml", "r") as file:
    fstring_template = file.read()

# replace variables and fix carriage returns
statistician_agent_specs = eval(f"f'''{fstring_template}'''")
print(statistician_agent_specs)

type: azure_assistant
name: Statistician
description: A helpful assistant that writes and executes code to process and analyze data.
model:
  id: gpt-4o
  options:
    temperature: 0.0
tools:
  - type: code_interpreter
instructions: | 
  # GOAL
  Extract:
  - Word count
  - Character count
  - Space count

  Then calculate:
  **MAGIC NUMBER = Word Count + Character Count + Space Count**

  This number simulates a way to measure the quality of a sentence.

  **MANDATORY RULES**
  - **NEVER** try to interpret the meaning or do the semantic analysis of the given text. Consider it as a meaningless string.
  - **NEVER** repeat the given joke, text or input.
  - Do NOT ask **ANY** questions, just follow **ALL** the steps below **IN A SINGLE SHOT**.
  - **ALWAYS** return THE MAGIC NUMBER.


## Create the assistant client

In [17]:
from semantic_kernel.agents import AzureAssistantAgent
assistant_client = AzureAssistantAgent.create_client()
print(f"Assistant base URL: {assistant_client.base_url}")

Assistant base URL: https://mmoaiswc-01.openai.azure.com/openai/


## Create the assistant agent

In [18]:
from semantic_kernel.agents import AgentRegistry

statistician_agent: AzureAIAgent = await AgentRegistry.create_from_yaml(
    yaml_str=statistician_agent_specs,
    client=assistant_client
)
statistician_agent

AzureAssistantAgent(arguments={'temperature': 0.0}, description='A helpful assistant that writes and executes code to process and analyze data.', id='asst_JwLS1w2f7qYctU4ca1a27CYc', instructions='# GOAL\nExtract:\n- Word count\n- Character count\n- Space count\n\nThen calculate:\n**MAGIC NUMBER = Word Count + Character Count + Space Count**\n\nThis number simulates a way to measure the quality of a sentence.\n\n**MANDATORY RULES**\n- **NEVER** try to interpret the meaning or do the semantic analysis of the given text. Consider it as a meaningless string.\n- **NEVER** repeat the given joke, text or input.\n- Do NOT ask **ANY** questions, just follow **ALL** the steps below **IN A SINGLE SHOT**.\n- **ALWAYS** return THE MAGIC NUMBER.', kernel=Kernel(retry_mechanism=PassThroughWithoutRetry(), services={}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x77dce3e65e50>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], au

## Invoke the agent

In [19]:
STATISTICIAN_USER_INPUTS = [
    "Why don't snakes use computers? Because they can't find the 'escape' key!",
]

await ChatWithAgentStreamAsync(statistician_agent, STATISTICIAN_USER_INPUTS)


************************************
Message 0 from AuthorRole.USER: 'Why don't snakes use computers? Because they can't find the 'escape' key!'


```python
# Define the text input
text_input = "Why don't snakes use computers? Because they can't find the 'escape' key!"

# Calculate word count
word_count = len(text_input.split())

# Calculate character count (excluding spaces)
character_count = len(text_input.replace(" ", ""))

# Calculate space count
space_count = text_input.count(" ")

# Calculate the magic number
magic_number = word_count + character_count + space_count

# Return the magic number
magic_number
```

# AuthorRole.ASSISTANT: The MAGIC NUMBER is 85.


# 5. [Responses Agent](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-types/responses-agent?pivots=programming-language-python) with plugin and streaming - `reviewer_agent`
The OpenAI Responses API is OpenAI's most advanced interface for generating model responses. It supports text and image inputs, and text outputs. You are able to create stateful interactions with the model, using the output of previous responses as input. It is also possible to extend the model's capabilities with built-in tools for file search, web search, computer use, and more.

- [OpenAI Responses API](https://platform.openai.com/docs/api-reference/responses)
- [Responses API in Azure](https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/responses?tabs=python-secure)

## Load the agent definition

In [20]:
# Read the agent template from the file
with open("./_agents/reviewer_cca.yaml", "r") as file: # use reviewer_cca.yaml if responses still has the bug
    fstring_template = file.read()

# replace variables and fix carriage returns
reviewer_agent_specs = eval(f"f'''{fstring_template}'''")
print(reviewer_agent_specs)

type: chat_completion_agent
name: Reviewer
description: Agent that reviews a sentence to check if it is satisfactory
model:
  id: gpt-4o
  options:
    temperature: 0.0
tools:
  - id: MagicNumberValidator.validate_magic_number
    type: function
instructions:
    # TASKS
    Check if the MAGIC NUMBER is present and validated.
    - If the MAGIC NUMBER is not present --> NOT SATISFIED.
    - If the MAGIC NUMBER is not validated --> NOT SATISFIED.
    - If the MAGIC NUMBER is validated --> SATISFIED.

    **REGARDELESS** of the magic number being provided, **ALWAYS** start you answer saying if you are satisfied or not, reporting **ALSO** the validated magic number, if provided.


## Set up the client and model using Azure OpenAI Resources

In [21]:
from semantic_kernel.agents import AzureResponsesAgent

os.environ["AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME"] =  deployment_name
revieweragent_client = AzureResponsesAgent.create_client()

## Plugin

In [22]:
class MagicNumberValidator:
    from typing import Annotated
    from semantic_kernel.functions import kernel_function

    def __init__(self):
        self.validation = False

    @kernel_function(
        name="validate_magic_number",
        description="Validates the magic number",
    )
    def validate_mn(
        self,
        number: str,
    ) -> Annotated[str, "Validates the magic number"]:
        try:
            num = int(number)
            if (num % 2  == 0):
                self.validation = f"Validation was SUCCESSFUL for number {num}."
            else:
                self.validation = f"Validation FAILED for number {num}."
        except:
            self.validation = f"Validation FAILED for value {number}."
                
            
        return self.validation

mnv = MagicNumberValidator()
mnv.validate_mn("96")

'Validation was SUCCESSFUL for number 96.'

## Create the Semantic Kernel Agent, based on Azure OpenAI Responses Agent

Here is the implementation of the AzureResponsesAgent that, at the moment Aug 15th 2025 has a bug that makes it extremely slow.

```
from semantic_kernel.agents import AzureResponsesAgent

reviewer_agent = await AzureResponsesAgent.from_yaml(
    yaml_str=reviewer_agent_specs,
    client=revieweragent_client,
    plugins=[MagicNumberValidator()],
)

# bug workaround
reviewer_agent.instructions = reviewer_agent.instruction_role
reviewer_agent.instruction_role = "developer"

reviewer_agent
```

In [23]:
# because of the above bug, here we implement this agent as a ChatCompletionAgent

reviewer_agent: AzureAIAgent = await AgentRegistry.create_from_yaml(
    yaml_str=reviewer_agent_specs,
    kernel=kernel,
    plugins=[MagicNumberValidator()],
)

reviewer_agent

ChatCompletionAgent(arguments={'temperature': 0.0}, description='Agent that reviews a sentence to check if it is satisfactory', id='beefde63-9d3b-4222-86ee-b551c47c922a', instructions='Check if the MAGIC NUMBER is present and validated. - If the MAGIC NUMBER is not present --> NOT SATISFIED. - If the MAGIC NUMBER is not validated --> NOT SATISFIED. - If the MAGIC NUMBER is validated --> SATISFIED.\n**REGARDELESS** of the magic number being provided, **ALWAYS** start you answer saying if you are satisfied or not, reporting **ALSO** the validated magic number, if provided.', kernel=Kernel(retry_mechanism=PassThroughWithoutRetry(), services={'chatcompletion_service_id': AzureChatCompletion(ai_model_id='gpt-4o', service_id='chatcompletion_service_id', instruction_role='system', client=<openai.lib.azure.AsyncAzureOpenAI object at 0x77dcf427d550>, ai_model_type=<OpenAIModelTypes.CHAT: 'chat'>, prompt_tokens=0, completion_tokens=0, total_tokens=0)}, ai_service_selector=<semantic_kernel.servic

## Invoke the Responses Agent

In [24]:
REVIEWER_USER_INPUTS = [
    "The magic number is 102", 
    "Today is a sunny day, and the magic nr is 103",
    "I will rain tomorrow",
]

await ChatWithAgentStreamAsync(reviewer_agent, REVIEWER_USER_INPUTS)


************************************
Message 0 from AuthorRole.USER: 'The magic number is 102'

# AuthorRole.ASSISTANT: SATISFIED. The validated magic number is 102.

************************************
Message 0 from AuthorRole.USER: 'Today is a sunny day, and the magic nr is 103'

# AuthorRole.ASSISTANT: NOT SATISFIED. The provided magic number 103 was not validated.

************************************
Message 0 from AuthorRole.USER: 'I will rain tomorrow'

# AuthorRole.ASSISTANT: NOT SATISFIED. There is no MAGIC NUMBER provided to validate.


# Group Chats

## [Sequential Orchestration](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-orchestration/sequential?pivots=programming-language-python)

In [25]:
def get_agents():
    agents = [creaturequestioner_agent, animalpicker_agent, animaljoker_agent, statistician_agent, reviewer_agent]
    return agents

### Observe Agent Responses
You can define a callback to observe and print the output from each agent as the sequence progresses.

In [26]:
from semantic_kernel.contents import ChatMessageContent

# Track the last agent name to avoid repeating it unnecessarily
last_agent_name = None
first_print = True

def agent_response_callback(message: ChatMessageContent) -> None:
    global last_agent_name
    global first_print

    # Print agent name only when it changes
    if message.name != last_agent_name:
        if first_print:
            first_print = False
        else:
            print(f"\n\n\n")

        if not message.name is None: # sometimes we get this nonsense agent, which we don't print
            print(f"==> AGENT **{message.name}**---\n", end="", flush=True)
            
        last_agent_name = message.name

    # Stream content inline
    print(message.content, end="", flush=True)

### Set Up the Sequential Orchestration
SequentialOrchestration object, passing in the agents and the optional response callback.

In [27]:
from semantic_kernel.agents import SequentialOrchestration

agents = get_agents()
sequential_orchestration = SequentialOrchestration(
    members=agents,
    agent_response_callback=agent_response_callback,
)

### Start the Runtime
Start the runtime to manage agent execution

In [28]:
from semantic_kernel.agents.runtime import InProcessRuntime

runtime = InProcessRuntime()
runtime.start()

==> AGENT **CreatureQuestioner**---
Which mammal is known for having the longest lifespan?



==> AGENT **AnimalPicker**---
Bowhead whale



==> AGENT **AnimalJoker**---
Here is the joke - Why did the bowhead whale always invite the fish to its party? It couldn't resist a good fin-ish!



==> AGENT **Statistician**---
# Given text
text = "Why did the bowhead whale always invite the fish to its party? It couldn't resist a good fin-ish!"

# Step-by-step calculations
word_count = len(text.split())
character_count = len(text)
space_count = text.count(" ")

# Calculating the MAGIC NUMBER
magic_number = word_count + character_count + space_count
magic_numberThe MAGIC NUMBER for the given text is 132.



==> AGENT **Reviewer**---
SATISFIED. The validated magic number is 132.

### Invoke the Orchestration
Invoke the orchestration with your initial task (e.g., a product description). The output will flow through each agent in sequence.

In [29]:
orchestration_result = await sequential_orchestration.invoke(
    task="An eco-friendly stainless steel water bottle that keeps drinks cold for 24 hours",
    runtime=runtime,
)

### Stop the Runtime
After processing is complete, stop the runtime to clean up resources.

In [30]:
await runtime.stop_when_idle()

### Collect Results
Wait for the orchestration to complete.

In [31]:
value = await orchestration_result.get(timeout=20)
print(f"***** Final Result *****\n{value}")

***** Final Result *****
SATISFIED. The validated magic number is 132.


# Teardown

In [32]:
# delete all files
files_to_delete = await project_client.agents.files.list()
files_to_delete_nr = len(files_to_delete.data)

if files_to_delete_nr>0:
    i=0
    print(f"{files_to_delete_nr} files will now be deleted:")
    for f in files_to_delete.data:
        i += 1
        print(f"- File {i} of {files_to_delete_nr}: {f.filename} (id={f.id}) is being deleted...")
        await project_client.agents.files.delete(f.id)
else:
    print("No files to delete")

No files to delete


## Avoiding ***modifying a collection while iterating over it*** for both threads and agents

The code
```
threads_to_delete = project_client.agents.threads.list()
```
returns an async iterator that **lazily** fetches pages of threads.<br/>
But since we're deleting threads as we iterate, the underlying data source is being mutated during iteration. So when the iterator tries to fetch the next page, it hits a missing resource — hence the **ResourceNotFoundError**.<br/><br/>

This is a classic case of *modifying a collection while iterating over it*, which is risky even in synchronous code — and doubly so in async paged APIs.
### The solution
We need to fully materialize the list of threads before deleting anything. That way, the iterator isn’t affected by the deletions

In [33]:
# delete all threads

threads_to_delete = [t async for t in project_client.agents.threads.list()]
i = 0
for t in threads_to_delete:
    i += 1
    print(f"{i} - Thread <{t.id}> is being deleted...")
    await project_client.agents.threads.delete(thread_id=t.id)

1 - Thread <thread_cj2ijjPE9Uo32lVmwTSU6UiT> is being deleted...
2 - Thread <thread_4X6O6TEEMBDSMUo1iFC047PL> is being deleted...
3 - Thread <thread_CInvLdZzN7Bgor50ycVfwwI5> is being deleted...
4 - Thread <thread_t0Tm1V0SF4ndgQXbWA952RKl> is being deleted...


In [34]:
# delete all agents

agents_to_delete = [a async for a in project_client.agents.list_agents(limit=100)]
i=0
for a in agents_to_delete:
    i += 1
    print(f"{i} - Agent <{a.id}> is being deleted...")
    await project_client.agents.delete_agent(agent_id=a.id)

1 - Agent <asst_OyHk2CMifd9EmUu89NniD6xZ> is being deleted...
